# FitClub ЛЗ6 — Monte Carlo мерзім болжамы

**Мақсат:** 15 story-ден тұратын MVP backlog-ының аяқталу уақытын 10 000 симуляциямен бағалау.  
**Дерек:** 6-дәрістегі оқытушы үлгісі: апталық throughput `[3, 8, 4, 10, 2, 7, 9, 5, 6, 6]`.  
**Шектеу:** бұл FitClub-тың нақты тарихы емес; әдісті көрсетуге арналған сыртқы reference class.


In [ ]:
import random
import math
from collections import Counter
from datetime import date, timedelta
import matplotlib.pyplot as plt

THROUGHPUT = [3, 8, 4, 10, 2, 7, 9, 5, 6, 6]
BACKLOG = 15
RUNS = 10_000
SEED = 42
START_DATE = date(2026, 10, 8)

print("Орташа throughput:", sum(THROUGHPUT) / len(THROUGHPUT))


In [ ]:
rng = random.Random(SEED)
completion_weeks = []

for _ in range(RUNS):
    done = 0
    weeks = 0
    while done < BACKLOG:
        done += rng.choice(THROUGHPUT)
        weeks += 1
    completion_weeks.append(weeks)

counts = Counter(completion_weeks)
distribution = [(w, counts[w], counts[w] / RUNS) for w in sorted(counts)]
distribution


In [ ]:
def percentile_higher(values, probability):
    ordered = sorted(values)
    index = max(0, math.ceil(probability * len(ordered)) - 1)
    return ordered[index]

for p in (0.50, 0.70, 0.85, 0.95):
    weeks = percentile_higher(completion_weeks, p)
    finish = START_DATE + timedelta(weeks=weeks)
    print(f"P{int(p*100)}: {weeks} апта — {finish:%d.%m.%Y}")


In [ ]:
weeks = [row[0] for row in distribution]
probabilities = [row[2] for row in distribution]

plt.figure(figsize=(8, 4.5))
plt.bar(weeks, probabilities, color="#167D9A")
plt.title("FitClub MVP аяқталу аптасының таралуы")
plt.xlabel("Апта саны")
plt.ylabel("Ықтималдық")
plt.gca().yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.grid(axis="y", alpha=.2)
plt.show()


In [ ]:
cumulative = []
running = 0
for week, _, probability in distribution:
    running += probability
    cumulative.append(running)

plt.figure(figsize=(8, 4.5))
plt.step(weeks, cumulative, where="post", color="#F59E0B", linewidth=3)
for level in (0.50, 0.70, 0.85, 0.95):
    plt.axhline(level, color="#94A3B8", linestyle="--", linewidth=.8)
plt.title("Жинақталған аяқталу ықтималдығы")
plt.xlabel("Апта саны")
plt.ylabel("P(аяқталу ≤ апта)")
plt.ylim(0, 1.05)
plt.gca().yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.grid(alpha=.2)
plt.show()


## Күтілетін нәтиже (seed = 42)

| Көрсеткіш | Нәтиже |
|---|---|
| P50 | 3 апта — 29.10.2026 |
| P70 | 3 апта — 29.10.2026 |
| P85 | 4 апта — 05.11.2026 |
| P95 | 4 апта — 05.11.2026 |

Таралу: 2 апта — 2 530; 3 апта — 5 345; 4 апта — 1 887; 5 апта — 228; 6 апта — 10.

## Басқарушылық түсіндіру

ЛЗ5-тегі baseline — 16.12.2026. Monte Carlo нәтижесі оны автоматты алмастырмайды, өйткені сыртқы үлгі story санын қолданады, ал FitClub жоспары story point, интеграция, тест және құжаттандыруды қамтиды. 22.10.2026 күні өз throughput дерегімізбен қайта есептейміз.
